# Project 2 — Credit-Card Fraud Detection

**Goal:** Explore an imbalanced fraud dataset, handle class imbalance, train tree-based models
(Random Forest / XGBoost), evaluate with precision/recall/ROC-AUC, and discuss the
precision-vs-recall business tradeoff with a decision threshold.

**Dataset:** This notebook is written for the classic Kaggle *Credit Card Fraud Detection* dataset
(`creditcard.csv`), which has anonymized PCA features `V1`...`V28`, plus `Time`, `Amount`,
and the target `Class` (1 = fraud, 0 = legit). Download it from Kaggle:
https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud and place `creditcard.csv` in the
same folder as this notebook (or update `DATA_PATH` below).


In [ ]:
# 1. Imports
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, roc_curve, roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score
)

# Optional: XGBoost (pip install xgboost if not already installed)
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed — run `pip install xgboost` to enable that section.")

# Optional: imbalanced-learn for SMOTE / under-sampling
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.pipeline import Pipeline as ImbPipeline
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False
    print("imbalanced-learn not installed — run `pip install imbalanced-learn` to enable SMOTE/undersampling.")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
RANDOM_STATE = 42


## 1. Load the data

In [ ]:
DATA_PATH = "creditcard.csv"  # <-- update this path if needed

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


## 2. Exploratory Data Analysis (EDA)

Basic structure, missing values, and summary statistics before we look at the class imbalance.


In [ ]:
df.info()


In [ ]:
df.isnull().sum().sum()  # total missing values across the dataset


In [ ]:
df.describe().T


### 2.1 Class imbalance — the core challenge of this dataset

In [ ]:
class_counts = df["Class"].value_counts()
class_pct = df["Class"].value_counts(normalize=True) * 100

print(class_counts)
print(class_pct.round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(x="Class", data=df, ax=axes[0], palette=["#4C72B0", "#C44E52"])
axes[0].set_title("Class counts (0 = Legit, 1 = Fraud)")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Count")
for i, v in enumerate(class_counts.sort_index()):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom")

axes[1].pie(
    class_counts, labels=["Legit", "Fraud"], autopct="%1.3f%%",
    colors=["#4C72B0", "#C44E52"], startangle=90
)
axes[1].set_title("Class proportion")

plt.tight_layout()
plt.show()

print(f"Fraud cases represent only {class_pct[1]:.4f}% of all transactions — a heavily imbalanced dataset.")


### 2.2 Distributions: Amount and Time by class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x="Class", y="Amount", data=df, ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Transaction Amount by Class (log scale)")

sns.histplot(data=df, x="Time", hue="Class", bins=50, stat="density",
             common_norm=False, ax=axes[1])
axes[1].set_title("Transaction Time distribution by Class")

plt.tight_layout()
plt.show()


### 2.3 Correlation heatmap (PCA features vs Class)

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.2)
plt.title("Correlation Heatmap")
plt.show()

# Features most correlated with the fraud label
corr["Class"].abs().sort_values(ascending=False).head(11)


## 3. Train / test split and scaling

We split **before** any resampling so the test set stays a realistic, untouched sample of
real-world class proportions. Only the *training* data will be resampled.


In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Scale Time and Amount (the PCA components V1-V28 are already roughly standardized)
scaler = StandardScaler()
for col in ["Time", "Amount"]:
    X_train[col] = scaler.fit_transform(X_train[[col]])
    X_test[col] = scaler.transform(X_test[[col]])

print("Train shape:", X_train.shape, " Fraud rate:", y_train.mean().round(5))
print("Test shape :", X_test.shape,  " Fraud rate:", y_test.mean().round(5))


## 4. Handling class imbalance

We compare three resampling strategies on the **training set only**:

1. **Random Under-sampling** — down-sample the majority (legit) class.
2. **SMOTE (over-sampling)** — synthesize new minority (fraud) examples.
3. **No resampling (baseline)** — rely on `class_weight="balanced"` instead.

Each strategy is later paired with a model so we can compare precision/recall tradeoffs.


In [ ]:
resampled_sets = {"Original (no resampling)": (X_train, y_train)}

if HAS_IMBLEARN:
    # Under-sampling
    rus = RandomUnderSampler(random_state=RANDOM_STATE)
    X_rus, y_rus = rus.fit_resample(X_train, y_train)
    resampled_sets["Random Under-sampling"] = (X_rus, y_rus)

    # SMOTE over-sampling
    smote = SMOTE(random_state=RANDOM_STATE)
    X_smote, y_smote = smote.fit_resample(X_train, y_train)
    resampled_sets["SMOTE"] = (X_smote, y_smote)

    for name, (Xr, yr) in resampled_sets.items():
        print(f"{name:28s} -> shape {Xr.shape}, fraud rate {yr.mean():.4f}")
else:
    print("Install imbalanced-learn to run this cell: pip install imbalanced-learn")


## 5. Model training — Random Forest & XGBoost

We train a model for each resampling strategy and keep the fitted models + predicted
probabilities for the shared evaluation section below.


In [ ]:
results = {}  # name -> dict(model, y_pred, y_proba)

def train_and_store(name, model, X_tr, y_tr):
    model.fit(X_tr, y_tr)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)  # default 0.5 threshold; tuned later
    results[name] = {"model": model, "y_proba": y_proba, "y_pred": y_pred}
    return model

# --- Random Forest, baseline (class_weight balances the imbalance) ---
rf_baseline = RandomForestClassifier(
    n_estimators=300, max_depth=None, class_weight="balanced",
    n_jobs=-1, random_state=RANDOM_STATE
)
train_and_store("RandomForest (class_weight=balanced)", rf_baseline, X_train, y_train)

# --- Random Forest, on resampled data (if imbalanced-learn available) ---
if HAS_IMBLEARN:
    rf_smote = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE)
    train_and_store("RandomForest + SMOTE", rf_smote, *resampled_sets["SMOTE"])

    rf_under = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE)
    train_and_store("RandomForest + UnderSampling", rf_under, *resampled_sets["Random Under-sampling"])

# --- XGBoost, using scale_pos_weight to handle imbalance ---
if HAS_XGB:
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    xgb_model = XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.1,
        scale_pos_weight=scale_pos_weight, eval_metric="aucpr",
        use_label_encoder=False, random_state=RANDOM_STATE, n_jobs=-1
    )
    train_and_store("XGBoost (scale_pos_weight)", xgb_model, X_train, y_train)

    if HAS_IMBLEARN:
        xgb_smote = XGBClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.1,
            eval_metric="aucpr", use_label_encoder=False,
            random_state=RANDOM_STATE, n_jobs=-1
        )
        train_and_store("XGBoost + SMOTE", xgb_smote, *resampled_sets["SMOTE"])

list(results.keys())


## 6. Evaluation — Precision, Recall, ROC-AUC

Accuracy is a misleading metric on a ~0.17% fraud rate dataset (predicting "all legit" would
already score >99.8% accuracy), so we focus on **precision, recall, F1, ROC-AUC, and
Average Precision (PR-AUC)** instead.


In [ ]:
summary_rows = []
for name, r in results.items():
    y_pred, y_proba = r["y_pred"], r["y_proba"]
    summary_rows.append({
        "Model": name,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "PR-AUC (Avg Precision)": average_precision_score(y_test, y_proba),
    })

summary_df = pd.DataFrame(summary_rows).set_index("Model").sort_values("PR-AUC (Avg Precision)", ascending=False)
summary_df.round(4)


In [ ]:
# Detailed classification report for the best model by PR-AUC
best_name = summary_df.index[0]
print(f"Best model by PR-AUC: {best_name}\n")
print(classification_report(y_test, results[best_name]["y_pred"], target_names=["Legit", "Fraud"]))


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, results[best_name]["y_pred"], display_labels=["Legit", "Fraud"],
    cmap="Blues", ax=ax, colorbar=False
)
ax.set_title(f"Confusion Matrix — {best_name}")
plt.show()


### 6.1 ROC curves

In [ ]:
plt.figure(figsize=(7, 6))
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r["y_proba"])
    auc = roc_auc_score(y_test, r["y_proba"])
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend(fontsize=8)
plt.show()


### 6.2 Precision-Recall curves

For heavily imbalanced problems like fraud detection, the **PR curve is usually more
informative than the ROC curve**, since ROC-AUC can look deceptively good even for a model
that's not very useful at picking out the rare fraud cases.


In [ ]:
plt.figure(figsize=(7, 6))
for name, r in results.items():
    prec, rec, _ = precision_recall_curve(y_test, r["y_proba"])
    ap = average_precision_score(y_test, r["y_proba"])
    plt.plot(rec, prec, label=f"{name} (AP={ap:.3f})")

baseline_rate = y_test.mean()
plt.axhline(baseline_rate, color="k", linestyle="--", label=f"No-skill baseline ({baseline_rate:.4f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves")
plt.legend(fontsize=8)
plt.show()


## 7. Precision vs Recall tradeoff & business decision threshold

In fraud detection, the confusion-matrix cells map to real costs:

- **False Positive** (legit flagged as fraud) → annoyed customer, blocked card, support cost.
- **False Negative** (fraud missed) → direct financial loss, possibly larger than a support cost.

Because missing fraud is usually far more expensive than a false alarm, businesses often
**deliberately lower the classification threshold below 0.5** to trade some precision for
higher recall. The cell below sweeps thresholds for the best model and lets you pick one
based on a minimum-recall or minimum-precision business rule.


In [ ]:
best_proba = results[best_name]["y_proba"]
prec, rec, thresh = precision_recall_curve(y_test, best_proba)

# Business rule example: choose the lowest threshold that still guarantees >= 90% recall
target_recall = 0.90
valid_idx = np.where(rec[:-1] >= target_recall)[0]
if len(valid_idx) > 0:
    chosen_idx = valid_idx[-1]  # highest precision among thresholds meeting the recall bar
    chosen_threshold = thresh[chosen_idx]
    print(f"To guarantee recall >= {target_recall:.0%}, use threshold = {chosen_threshold:.4f}")
    print(f"  -> Precision at this threshold: {prec[chosen_idx]:.4f}, Recall: {rec[chosen_idx]:.4f}")
else:
    chosen_threshold = 0.5
    print("No threshold reaches the target recall; falling back to 0.5.")

y_pred_business = (best_proba >= chosen_threshold).astype(int)
print("\nClassification report at the business threshold:")
print(classification_report(y_test, y_pred_business, target_names=["Legit", "Fraud"]))


In [ ]:
# Visualize precision & recall as a function of threshold, with the chosen cutoff marked
plt.figure(figsize=(8, 5))
plt.plot(thresh, prec[:-1], label="Precision")
plt.plot(thresh, rec[:-1], label="Recall")
plt.axvline(chosen_threshold, color="red", linestyle="--",
            label=f"Chosen threshold = {chosen_threshold:.3f}")
plt.xlabel("Decision threshold")
plt.ylabel("Score")
plt.title(f"Precision & Recall vs Threshold — {best_name}")
plt.legend()
plt.show()


### 7.1 Discussion — presenting tradeoffs to the business

- **High-precision threshold (closer to 1.0):** fewer false alarms, better customer
  experience, but more fraud slips through undetected — appropriate when manual review
  capacity is limited or false positives are very costly (e.g., blocking a VIP customer).
- **High-recall threshold (closer to 0.0):** catches nearly all fraud, but floods the fraud
  operations team with false positives to review — appropriate when the average fraud loss
  is large relative to the cost of a manual review.
- **Recommended approach:** present the business with the precision/recall table at 2-3
  candidate thresholds (e.g., 90%, 95%, 98% recall) along with the *expected number of daily
  false positives* at each, and let them pick based on operational review capacity and the
  average dollar loss per missed fraud case.


In [ ]:
# Example: table of a few candidate thresholds for a stakeholder-facing summary
candidate_recalls = [0.99, 0.95, 0.90, 0.80]
rows = []
for target in candidate_recalls:
    idx = np.where(rec[:-1] >= target)[0]
    if len(idx) == 0:
        continue
    i = idx[-1]
    rows.append({
        "Target Recall": target,
        "Threshold": round(thresh[i], 4),
        "Actual Recall": round(rec[i], 4),
        "Actual Precision": round(prec[i], 4),
    })

pd.DataFrame(rows)


## 8. Summary

- The dataset is extremely imbalanced (~0.17% fraud), so accuracy is not a useful metric —
  precision, recall, PR-AUC, and ROC-AUC were used instead.
- Random under-sampling, SMOTE, and class-weighting were compared as ways to handle the
  imbalance before/while training Random Forest and XGBoost models.
- The best model was selected by **PR-AUC**, since it's the most informative metric for rare
  positive classes.
- The final threshold is a **business decision**, not a purely statistical one — it should be
  set jointly with stakeholders based on the relative cost of false positives vs false
  negatives.
